<a href="https://colab.research.google.com/github/EsatovinVictoria/ML_Semester5_EsatovinEbenhaezerVictoria/blob/main/Pertemuan_6/Praktikum_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Praktikum 4

Percobaan kali ini kita akan melihat perbedaan ketiga model yang telah kita bahas dan bandingkan hasilnya.

In [1]:
%pip install --no-cache-dir numpy pandas annoy faiss-cpu hnswlib

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 647.5/647.5 kB 15.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 161.2 MB/s eta 0:00:00
  Created wheel for annoy: filename=annoy-1.17.3-cp313-cp313-linux_x86_64.whl size=520037 sha256=46e41858d288ddc9f9dee6234be880f1961ae1ebb3258aea3ee3312c8b361e86
  Stored in directory: /tmp/pip-ephem-wheel-cache-vz880_1l/wheels/fc/4e/87/f1f957e7382aa370452cff98276f51686924d5415e449800ed
  Created wheel for hnswlib: filename=hnswlib-0.8.0-cp313-cp313-linux_x86_64.whl size=2679800 sha256=869c32f75888c37626bac1732b71eca911bf15197d7ac1911362dd7fa2625f4e
  Stored in directory: /tmp/pip-ephem-wheel-cache-vz880_1l/wheels/35/04/88/b31765a4b9957705e18065db4657e61fc8da54f50e3ef0b67e
Successfully built annoy hnswlib


In [2]:
import numpy as np
import time
from annoy import AnnoyIndex
import faiss
import hnswlib

# ===============================
# 1. Buat dataset 1 juta data 5D
# ===============================
n_data = 1_000_000   # bisa coba 100_000 dulu jika RAM terbatas
dim = 5
X = np.random.random((n_data, dim)).astype(np.float32)

# Query point
query = np.random.random((1, dim)).astype(np.float32)
k = 10

hasil = []

# ===============================
# 2. Annoy
# ===============================
print("=== Annoy ===")
ann_index = AnnoyIndex(dim, 'euclidean')

start = time.time()
for i in range(n_data):
    ann_index.add_item(i, X[i])
ann_index.build(10)  # 10 trees
build_time = time.time() - start

start = time.time()
neighbors = ann_index.get_nns_by_vector(query[0], k, include_distances=True)
query_time = time.time() - start

print("Build time:", build_time, "detik")
print("Query time:", query_time, "detik")
print("Neighbors:", neighbors[0][:5], "...")
hasil.append(["Euclidean", "Annoy", build_time, query_time, list(neighbors[0][:5])])

# ===============================
# 3. FAISS (Flat Index)
# ===============================
print("\n=== FAISS (IndexFlatL2) ===")
faiss_index = faiss.IndexFlatL2(dim)

start = time.time()
faiss_index.add(X)
build_time = time.time() - start

start = time.time()
distances, indices = faiss_index.search(query, k)
query_time = time.time() - start

print("Build time:", build_time, "detik")
print("Query time:", query_time, "detik")
print("Neighbors:", indices[0][:5], "...")
hasil.append(["Euclidean", "FAISS", build_time, query_time, list(indices[0][:5])])

# ===============================
# 4. HNSW (hnswlib)
# ===============================
print("\n=== HNSW (hnswlib) ===")
hnsw_index = hnswlib.Index(space='l2', dim=dim)

start = time.time()
hnsw_index.init_index(max_elements=n_data, ef_construction=200, M=16)
hnsw_index.add_items(X)
build_time = time.time() - start

hnsw_index.set_ef(50)

start = time.time()
labels, distances = hnsw_index.knn_query(query, k=k)
query_time = time.time() - start

print("Build time:", build_time, "detik")
print("Query time:", query_time, "detik")
print("Neighbors:", labels[0][:5], "...")
hasil.append(["Euclidean", "HNSW", build_time, query_time, list(labels[0][:5])])


=== Annoy ===
Build time: 30.860908269882202 detik
Query time: 0.00024771690368652344 detik
Neighbors: [132512, 820346, 764903, 746001, 924533] ...

=== FAISS (IndexFlatL2) ===
Build time: 0.016645431518554688 detik
Query time: 0.012615680694580078 detik
Neighbors: [132512 820346 764903 746001 924533] ...

=== HNSW (hnswlib) ===
Build time: 184.81357216835022 detik
Query time: 0.0002548694610595703 detik
Neighbors: [132512 820346 764903 746001 924533] ...


Lakukan percobaan pada metric distance yang berbeda. catat hasilnya pada tabel yang anda buat sendiri seperti pada praktikum 1.

### Percobaan Cosine

In [3]:
# ===============================
# 2. Annoy
# ===============================
print("=== Annoy ===")
ann_index = AnnoyIndex(dim, 'angular')

start = time.time()
for i in range(n_data):
    ann_index.add_item(i, X[i])
ann_index.build(10)  # 10 trees
build_time = time.time() - start

start = time.time()
neighbors = ann_index.get_nns_by_vector(query[0], k, include_distances=True)
query_time = time.time() - start

print("Build time:", build_time, "detik")
print("Query time:", query_time, "detik")
print("Neighbors:", neighbors[0][:5], "...")
hasil.append(["Cosine", "Annoy", build_time, query_time, list(neighbors[0][:5])])

# ===============================
# 3. FAISS (Flat Index)
# ===============================
print("\n=== FAISS (Cosine) ===")
X_normalized = X.copy()
query_normalized = query.copy()
faiss.normalize_L2(X_normalized)
faiss.normalize_L2(query_normalized)
faiss_index = faiss.IndexFlatIP(dim)

start = time.time()
faiss_index.add(X_normalized)
build_time = time.time() - start

start = time.time()
distances, indices = faiss_index.search(query_normalized, k)
query_time = time.time() - start

print("Build time:", build_time, "detik")
print("Query time:", query_time, "detik")
print("Neighbors:", indices[0][:5], "...")
hasil.append(["Cosine", "FAISS", build_time, query_time, list(indices[0][:5])])

# ===============================
# 4. HNSW (hnswlib)
# ===============================
print("\n=== HNSW (hnswlib) ===")
hnsw_index = hnswlib.Index(space='cosine', dim=dim)

start = time.time()
hnsw_index.init_index(max_elements=n_data, ef_construction=200, M=16)
hnsw_index.add_items(X)
build_time = time.time() - start

hnsw_index.set_ef(50)

start = time.time()
labels, distances = hnsw_index.knn_query(query, k=k)
query_time = time.time() - start

print("Build time:", build_time, "detik")
print("Query time:", query_time, "detik")
print("Neighbors:", labels[0][:5], "...")
hasil.append(["Cosine", "HNSW", build_time, query_time, list(labels[0][:5])])


=== Annoy ===
Build time: 37.72815990447998 detik
Query time: 0.0003135204315185547 detik
Neighbors: [700184, 871957, 220775, 218936, 967555] ...

=== FAISS (Cosine) ===
Build time: 0.018726110458374023 detik
Query time: 0.004297971725463867 detik
Neighbors: [700184 871957 220775 218936 967555] ...

=== HNSW (hnswlib) ===
Build time: 174.40255331993103 detik
Query time: 0.0002689361572265625 detik
Neighbors: [700184 871957 220775 218936 967555] ...


### Tabel Hasil Percobaan

In [4]:
import pandas as pd

pd.DataFrame(hasil, columns=[
    "Metrik", "Model", "Build Time (detik)", "Query Time (detik)", "5 Neighbors"
])

,Metrik,Model,Build Time (detik),Query Time (detik),5 Neighbors
0,Euclidean,Annoy,30.860908,0.000248,"[132512, 820346, 764903, 746001, 924533]"
1,Euclidean,FAISS,0.016645,0.012616,"[132512, 820346, 764903, 746001, 924533]"
2,Euclidean,HNSW,184.813572,0.000255,"[132512, 820346, 764903, 746001, 924533]"
3,Cosine,Annoy,37.728160,0.000314,"[700184, 871957, 220775, 218936, 967555]"
4,Cosine,FAISS,0.018726,0.004298,"[700184, 871957, 220775, 218936, 967555]"
5,Cosine,HNSW,174.402553,0.000269,"[700184, 871957, 220775, 218936, 967555]"
